[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/26_lora_solution.ipynb)

# 🟡 Solution: LoRA (Low-Rank Adaptation)

*Attention & Transformers · Medium*

Reference implementation. Try it yourself in `26_lora.ipynb` first.

---
Implement **LoRA**: freeze a pretrained linear layer and learn a low-rank
correction beside it.

$$h = W_0 x + \frac{\alpha}{r}\,(B A) x, \qquad
A \in \mathbb{R}^{d_{in} \times r},\; B \in \mathbb{R}^{r \times d_{out}}$$

### Signature
```python
class LoRALinear(nnx.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0,
                 *, rngs: nnx.Rngs): ...
    def __call__(self, x): ...
```

### Requirements
- `self.linear`: an `nnx.Linear(in_features, out_features)` that is **frozen**
- `self.lora_A`: small random, shape `(in_features, rank)`
- `self.lora_B`: **zeros**, shape `(rank, out_features)`
- `self.scaling = alpha / rank`
- Only `lora_A` and `lora_B` are trainable

### Why B starts at zero
At initialization $BA = 0$, so the adapted layer is **exactly** the pretrained
one. Fine-tuning therefore starts from the pretrained function rather than a
perturbed one — no warmup, no risk of destroying the base model on step one.
If both $A$ and $B$ were zero the gradient would also be zero and nothing would
ever learn, so exactly one of them is zeroed: random $A$ gives $B$ something to
receive gradient through.

### The parameter arithmetic
A $4096 \times 4096$ layer holds 16.8M weights. With $r = 8$ the adapter holds
$4096 \times 8 + 8 \times 4096 = 65{,}536$ — about **0.4%**. That is what makes
it possible to keep dozens of task-specific adapters for one base model, and to
train on a single GPU: the optimizer state, which for Adam is 2 floats per
trainable parameter, shrinks by the same factor.

### What $\alpha/r$ is for
It decouples the learning rate from the rank. Without it, doubling $r$ roughly
doubles the update magnitude and you would have to retune the learning rate
every time. With the scaling, $r$ becomes a capacity knob you can sweep freely.

### At inference, LoRA is free
$W_0 + \frac{\alpha}{r}BA$ can be folded into a single matrix once training
ends, so a merged adapter costs exactly nothing at serving time. The detour
only exists while you are training.

### ⚠️ A JAX layout note
PyTorch stores `lora_A` as `(rank, in_features)` and computes `x @ A.T @ B.T`.
Flax kernels are `(in, out)`, so here `A` is `(in_features, rank)` and `B` is
`(rank, out_features)` and the forward pass is a plain `x @ A @ B` — the same
maths, transposed to match the surrounding convention.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class LoRALinear(nnx.Module):
    def __init__(self, in_features: int, out_features: int, rank: int,
                 alpha: float = 1.0, *, rngs: nnx.Rngs):
        self.linear = nnx.Linear(in_features, out_features, rngs=rngs)

        # Freeze the base: demote its Params to plain Variables so
        # nnx.state(self, nnx.Param) — what an optimizer filters on — sees only
        # the adapter. This is the JAX counterpart of requires_grad_(False).
        self.linear.kernel = nnx.Variable(self.linear.kernel[...])
        self.linear.bias = nnx.Variable(self.linear.bias[...])

        # A random, B zero -> B @ A == 0 -> the adapter starts as a no-op.
        self.lora_A = nnx.Param(
            jax.random.normal(rngs.params(), (in_features, rank)) * 0.01
        )
        self.lora_B = nnx.Param(jnp.zeros((rank, out_features)))
        self.scaling = alpha / rank

    def __call__(self, x):
        return self.linear(x) + (x @ self.lora_A[...] @ self.lora_B[...]) * self.scaling

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

layer = LoRALinear(64, 64, rank=4, alpha=8.0, rngs=nnx.Rngs(params=0))
x = jax.random.normal(jax.random.key(1), (2, 64))

base = layer.linear(x)
print("adapter is a no-op at init:", bool(jnp.allclose(layer(x), base)))

trainable = sum(v.size for v in jax.tree.leaves(nnx.state(layer, nnx.Param)))
print(f"trainable: {trainable} of {64*64 + 64} base weights "
      f"({100*trainable/(64*64+64):.1f}%)")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("lora")